In [49]:
print("Training process for Random Forest...\n\n🛢️ Random Forest Pipeline: SMOTE -> Random Forest")
rf_pipeline = ImbPipeline([
    (
        "smote", 
        SMOTE(random_state=42)
    ),
    (
        "model",
        RandomForestClassifier(
            random_state=42,
            n_jobs=-1
        )
    )
])
print("✅ Done!\n")

print("🔎 Random Forest hyperparameter search...")
rf_params = {
    "model__n_estimators": randint(150, 700),
    "model__max_depth": [10, 15, 20, 25, 30, None],
    "model__min_samples_split": randint(5, 50),
    "model__min_samples_leaf": randint(2, 15),
    "model__max_features": ["sqrt", "log2", 0.3, 0.5]
}
print("✅ Done!\n")

print("🚀 Initializing cross-validation object...")
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
print("✅ Done!\n")

print("🎲 Initializing a search manager object...")
rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_params,
    n_iter = 30,
    scoring="roc_auc",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)
print("✅ Done!\n")

Training process for Random Forest...

🛢️ Random Forest Pipeline: SMOTE -> Random Forest
✅ Done!

🔎 Random Forest hyperparameter search...
✅ Done!

🚀 Initializing cross-validation object...
✅ Done!

🎲 Initializing a search manager object...
✅ Done!



In [30]:
print("🎓 Fitting the search manager...")
rf_search.fit(X_train_rf, y_train_rf)

print(f"Best hyperparameters for Random Forest model are:\n{rf_search.best_params_}")
print(f"The best score is:{rf_search.best_score_}\n")

best_rf = rf_search.best_estimator_
print("💾 Saving best performing Random Forest model...")

try:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    joblib.dump(best_rf, f"../models/random_forest_best_{timestamp}.joblib")
    joblib.dump(rf_search, f"../models/rf_random_search_{timestamp}.joblib")
    print("✅ Random Forest model with best performance score was successfully saved!")
    
except:
    print("❌ Something went wrong during the model serialization...Check Traceback call")

🎓 Fitting the search manager...
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Best hyperparameters for Random Forest model are:
{'model__max_depth': 25, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 12, 'model__min_samples_split': 12, 'model__n_estimators': 170}
The best score is:0.9959988922878997

💾 Saving best performing Random Forest model...
✅ Random Forest model with best performance score was successfully saved!


In [31]:
from sklearn.ensemble import IsolationForest

print("📚 Train Isolation Forest on training period (time_step 1-34)\n")
if_model = IsolationForest(
    n_estimators=300,
    max_samples=2048,
    contamination=0.05,
    max_features=0.5,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)


if_model.fit(X_train_if)


print("🌱 Isolation Forest trained")

📚 Train Isolation Forest on training period (time_step 1-34)

🌱 Isolation Forest trained


In [32]:
print("💡 Create prediction helper function")
def create_if_predictions(
    model,
    X,
    df
):
    """
    Generate Isolation Forest outputs.
    """
    
    predictions = pd.DataFrame({
        "txId": df["txId"].values,
        "time_step": df["time_step"].values,

        # sklearn score:
        # higher = more normal
        # lower = more abnormal
        "if_score": model.score_samples(X),

        # sklearn decision:
        # negative = anomaly
        # positive = normal
        "decision_function": model.decision_function(X),

        # binary output
        "prediction": (
            model.predict(X) == -1
        ).astype(int)
    })


    return predictions

print("🆕 Prediction helper function created!")

💡 Create prediction helper function
🆕 Prediction helper function created!


In [55]:
val_predictions_df = create_if_predictions(
    if_model,
    X_val_if,
    validation_if_df
)


val_predictions_df.head()
print("🔄 Generated validation predictions\n")

val_eval_df = val_predictions_df.merge(
    validation_labels_if,
    on="txId",
    how="inner"
)


print(
    "🕵🏻 Validation labeled transactions:",
    len(val_eval_df)
)

val_eval_df.head()
print("✅ Created validation evaluation dataset")

🔄 Generated validation predictions

🕵🏻 Validation labeled transactions: 7829
✅ Created validation evaluation dataset


In [22]:
y_true = val_eval_df["is_illicit"]

y_score = val_eval_df["if_score"]


print("==============================")
print("Validation IF ranking")
print("==============================")


print(
    "ROC-AUC:",
    roc_auc_score(
        y_true,
        y_score
    )
)


print(
    "PR-AUC:",
    average_precision_score(
        y_true,
        y_score
    )
)

Validation IF ranking
ROC-AUC: 0.9070769007755307
PR-AUC: 0.6219264379570125


In [35]:
print("👩🏻‍🍳 Prepare final training data (time_step 1-41)")
X_train_if_full = (
    unsupervised_full_df[
        unsupervised_full_df["time_step"] <= 41
    ]
    .drop(
        columns=["txId", "time_step"]
    )
)


print(
    "🏁 Final training shape:",
    X_train_if_full.shape
)


👩🏻‍🍳 Prepare final training data (time_step 1-41)
🏁 Final training shape: (166945, 165)


In [53]:
final_if_model = IsolationForest(
    n_estimators=300,
    max_samples=2048,
    contamination=0.05,
    max_features=0.5,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)


final_if_model.fit(
    X_train_if_full
)

joblib.dump(final_if_model, "models/isolation_forest/isolation_forest_best.joblib")
print("👩🏻‍💻📓✍🏻💡 Final IF trained on time steps 1-41!")

print("🖇️ Merge test labels for evaluation\n")
test_eval_df = test_predictions_df.merge(
    test_labels_if,
    on="txId",
    how="inner"
)


print(
    "🧾 Test labeled transactions:",
    len(test_eval_df)
)

👩🏻‍💻📓✍🏻💡 Final IF trained on time steps 1-41!
🖇️ Merge test labels for evaluation

🧾 Test labeled transactions: 8841


In [63]:
print("❯❯❯❯ Creating the LOF pipeline...❯❯❯❯")

lof_pipeline = Pipeline([
    (
        "scaler", StandardScaler()
    ),
    (
        "lof", LocalOutlierFactor(
            n_neighbors=50,
            contamination=0.05,
            novelty=True,
            metric="euclidean",
            n_jobs=-1
        )
    )
])

print("✅ LOF pipeline created")


❯❯❯❯ Creating the LOF pipeline...❯❯❯❯
✅ LOF pipeline created


In [64]:
print("🧠 Training baseline LOF...")

lof_pipeline.fit(X_train_lof)

print("✅ LOF trained")

🧠 Training baseline LOF...
✅ LOF trained


In [66]:
print("👷 Generating validation scores...")
val_lof_predictions_df = pd.DataFrame({
    "txId": validation_if_df["txId"].values,
    "time_step": validation_if_df["time_step"].values,

    # sklearn LOF score:
    # higher = more normal
    # lower = more abnormal
    "lof_score": lof_pipeline.score_samples(X_val_lof),

    # sklearn LOF decision:
    # positive = normal
    # negative = anomaly
    "decision_function": lof_pipeline.decision_function(X_val_lof),

    # sklearn binary prediction
    "prediction": (
        lof_pipeline.predict(X_val_lof) == -1
    ).astype(int)
})

print(
    "Validation transactions scored:",
    len(val_lof_predictions_df)
)

val_lof_predictions_df.head()

👷 Generating validation scores...
Validation transactions scored: 30680


,txId,time_step,lof_score,decision_function,prediction
0,54785412,35,-1.095065,0.793099,0
1,69354384,35,-1.159707,0.728457,0
2,54775772,35,-1.472448,0.415717,0
3,69343934,35,-2.125604,-0.237440,1
4,70102750,35,-1.419554,0.468611,0


In [68]:
print("🏭 Merging validation labels")

val_lof_eval_df = val_lof_predictions_df.merge(
    validation_labels_if,
    on="txId",
    how="inner"
)

print("Validation labeled transactions:", len(val_lof_eval_df))

print("\nClass distribution:")

print(val_lof_eval_df["is_illicit"].value_counts())

🏭 Merging validation labels
Validation labeled transactions: 7829

Class distribution:
is_illicit
0    7154
1     675
Name: count, dtype: int64


In [71]:
print("⚖️ Baseline LOF ranking evaluation")
y_val_true = val_lof_eval_df["is_illicit"].astype(int)

# LOF:
# higher = more normal
# lower = more abnormal
#
# Therefore invert it:
y_val_anomaly_score = -val_lof_eval_df["lof_score"]

print("==============================")
print("Baseline LOF validation")
print("==============================")

print("Labeled transactions:", len(val_lof_eval_df))

print("ROC-AUC:", roc_auc_score(y_val_true, y_val_anomaly_score))
print("PR-AUC:", average_precision_score(y_val_true, y_val_anomaly_score))

⚖️ Baseline LOF ranking evaluation
Baseline LOF validation
Labeled transactions: 7829
ROC-AUC: 0.4738272295219458
PR-AUC: 0.07629340804963017


In [73]:
print("📈 Baseline binary prediction")
y_val_pred = val_lof_eval_df["prediction"].astype(int)

print("==============================")
print("Baseline LOF classification")
print("==============================")

print(
    "Precision:",
    precision_score(
        y_val_true,
        y_val_pred,
        zero_division=0
    )
)

print(
    "Recall:",
    recall_score(
        y_val_true,
        y_val_pred,
        zero_division=0
    )
)

print(
    "F1:",
    f1_score(
        y_val_true,
        y_val_pred,
        zero_division=0
    )
)

print("\nConfusion matrix:")

print(
    confusion_matrix(
        y_val_true,
        y_val_pred
    )
)

print("\nClassification report:")

print(
    classification_report(
        y_val_true,
        y_val_pred,
        digits=4,
        zero_division=0
    )
)

📈 Baseline binary prediction
Baseline LOF classification
Precision: 0.07258460029426189
Recall: 0.21925925925925926
F1: 0.10906411201179071

Confusion matrix:
[[5263 1891]
 [ 527  148]]

Classification report:
              precision    recall  f1-score   support

           0     0.9090    0.7357    0.8132      7154
           1     0.0726    0.2193    0.1091       675

    accuracy                         0.6911      7829
   macro avg     0.4908    0.4775    0.4611      7829
weighted avg     0.8369    0.6911    0.7525      7829



In [74]:
print("Anomaly transactions LOF produced:\n")

print("==============================")
print("LOF anomaly counts")
print("==============================")

print(
    "Anomalies in whole validation period:",
    int(
        (val_lof_predictions_df["prediction"] == 1).sum()
    )
)

print(
    "Total validation transactions:",
    len(val_lof_predictions_df)
)

print(
    "Anomaly rate in whole validation period:",
    (
        val_lof_predictions_df["prediction"].mean()
    )
)

Anomaly transactions LOF produced:

LOF anomaly counts
Anomalies in whole validation period: 4761
Total validation transactions: 30680
Anomaly rate in whole validation period: 0.1551825293350717


In [75]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

y_true = val_lof_eval_df["is_illicit"].astype(int)

lof_score = val_lof_eval_df["lof_score"]

print("==============================")
print("LOF score direction check")
print("==============================")

print("\nUsing LOF score:")
print(
    "ROC-AUC:",
    roc_auc_score(y_true, lof_score)
)
print(
    "PR-AUC:",
    average_precision_score(y_true, lof_score)
)

print("\nUsing inverted LOF score:")
print(
    "ROC-AUC:",
    roc_auc_score(y_true, -lof_score)
)
print(
    "PR-AUC:",
    average_precision_score(y_true, -lof_score)
)

LOF score direction check

Using LOF score:
ROC-AUC: 0.5261727704780541
PR-AUC: 0.09803068668059148

Using inverted LOF score:
ROC-AUC: 0.4738272295219458
PR-AUC: 0.07629340804963017


In [76]:
print("=" * 60)
print("🌐 LOF HYPERPARAMETER SEARCH")
print("=" * 60)


# ------------------------------------------------------------
# 1. Define parameter grid
# ------------------------------------------------------------

lof_param_grid = {
    "n_neighbors": [10, 20, 35, 50, 75, 100, 150],
    "contamination": [0.01, 0.03, 0.05, 0.08, 0.10, 0.15],
    "metric": ["euclidean", "manhattan"],
    "p": [1, 2],
    "leaf_size": [20, 30, 50],
}


# ------------------------------------------------------------
# 2. Generate combinations
# ------------------------------------------------------------

all_combinations = list(
    product(
        lof_param_grid["n_neighbors"],
        lof_param_grid["contamination"],
        lof_param_grid["metric"],
        lof_param_grid["p"],
        lof_param_grid["leaf_size"]
    )
)

print(
    "Total possible combinations:",
    len(all_combinations)
)


# ------------------------------------------------------------
# 3. Randomly select manageable number
# ------------------------------------------------------------

rng = np.random.default_rng(42)

n_iter = 50

selected_indices = rng.choice(
    len(all_combinations),
    size=min(n_iter, len(all_combinations)),
    replace=False
)

selected_combinations = [
    all_combinations[i]
    for i in selected_indices
]

print(
    "Configurations to evaluate:",
    len(selected_combinations)
)


# ------------------------------------------------------------
# 4. Prepare validation labels
# ------------------------------------------------------------

validation_eval_labels = validation_labels_if[
    ["txId", "is_illicit"]
].copy()


# ------------------------------------------------------------
# 5. Search
# ------------------------------------------------------------

lof_results = []


for i, params in enumerate(
    selected_combinations,
    start=1
):

    (
        n_neighbors,
        contamination,
        metric,
        p,
        leaf_size
    ) = params


    print(
        f"\n[{i}/{len(selected_combinations)}] "
        f"n_neighbors={n_neighbors}, "
        f"contamination={contamination}, "
        f"metric={metric}, "
        f"p={p}, "
        f"leaf_size={leaf_size}"
    )


    # --------------------------------------------------------
    # Train LOF
    # --------------------------------------------------------

    lof_model = LocalOutlierFactor(
        n_neighbors=n_neighbors,
        contamination=contamination,
        metric=metric,
        p=p,
        leaf_size=leaf_size,
        novelty=True,
        n_jobs=-1
    )


    lof_model.fit(X_train_lof)


    # --------------------------------------------------------
    # Score entire validation period
    # --------------------------------------------------------

    # LOF's decision_function:
    # higher = more normal
    #
    # We invert it:
    # higher = more anomalous

    validation_anomaly_score = (
        -lof_model.decision_function(X_val_lof)
    )


    validation_predictions = pd.DataFrame({
        "txId": validation_if_df["txId"].values,
        "anomaly_score": validation_anomaly_score
    })


    # --------------------------------------------------------
    # Keep only labeled validation transactions
    # --------------------------------------------------------

    evaluation_df = validation_predictions.merge(
        validation_eval_labels,
        on="txId",
        how="inner"
    )


    y_true = (
        evaluation_df["is_illicit"]
        .astype(int)
        .to_numpy()
    )


    y_score = (
        evaluation_df["anomaly_score"]
        .astype(float)
        .to_numpy()
    )


    # --------------------------------------------------------
    # Ranking metrics
    # --------------------------------------------------------

    roc_auc = roc_auc_score(
        y_true,
        y_score
    )


    pr_auc = average_precision_score(
        y_true,
        y_score
    )


    # --------------------------------------------------------
    # Binary prediction
    # --------------------------------------------------------

    # LOF's own prediction:
    # -1 = anomaly
    #  1 = normal

    raw_prediction = lof_model.predict(
        X_val_lof
    )


    y_pred_all = (
        raw_prediction == -1
    ).astype(int)


    validation_binary_predictions = pd.DataFrame({
        "txId": validation_if_df["txId"].values,
        "prediction": y_pred_all
    })


    binary_eval_df = validation_binary_predictions.merge(
        validation_eval_labels,
        on="txId",
        how="inner"
    )


    y_pred = (
        binary_eval_df["prediction"]
        .astype(int)
        .to_numpy()
    )


    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred
    ).ravel()


    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    lof_results.append({

        "n_neighbors": n_neighbors,
        "contamination": contamination,
        "metric": metric,
        "p": p,
        "leaf_size": leaf_size,

        "labeled_validation_transactions":
            len(evaluation_df),

        "predicted_anomalies_whole_period":
            int(y_pred_all.sum()),

        "roc_auc": roc_auc,
        "pr_auc": pr_auc,

        "precision":
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    })


# ============================================================
# 6. Results DataFrame
# ============================================================

lof_results_df = pd.DataFrame(
    lof_results
)


# ------------------------------------------------------------
# Sort primarily by PR-AUC
# ------------------------------------------------------------

lof_results_df = (
    lof_results_df
    .sort_values(
        by=[
            "pr_auc",
            "roc_auc",
            "recall",
            "f1"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n")
print("=" * 60)
print("TOP LOF CONFIGURATIONS")
print("=" * 60)


display(
    lof_results_df.head(15)
)


# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

lof_results_df.to_csv(
    "lof_hyperparameter_search_results.csv",
    index=False
)


print(
    "\nResults saved to:",
    "lof_hyperparameter_search_results.csv"
)

🌐 LOF hyperparameter search
